# Ingest results file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
   - Source File
   - Ingestion Timestamp
3. Write to bronze delta table  

In [0]:
%run "../00-common/01.environment-config"


In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
val source_file= landing_folder_path + "/results"
val table_name= catalog_name + "." + bronze_schema + "." + "results"

In [0]:
import org.apache.spark.sql.types.{StructType,StructField,StringType,DateType,IntegerType,FloatType}

val results_schema = StructType(Seq(
    StructField("date", DateType),
    StructField("raceName", StringType),
    StructField("round", IntegerType),
    StructField("season", IntegerType),
    StructField("url", StringType),
    StructField("constructorId", StringType),
    StructField("driverId", StringType),
    StructField("grid", IntegerType),
    StructField("laps", IntegerType),
    StructField("number", IntegerType),
    StructField("points", FloatType),
    StructField("position", IntegerType),
    StructField("positionText", StringType),
    StructField("status", StringType)
))


val results_df=spark.read.format("json")
.option("mode", "FAILFAST")
.schema(results_schema)
.load(source_file)

display(results_df)

In [0]:

val results_final_df= add_ingestion_metadata(results_df)

display(results_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
results_final_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))